# AIOps – Google Cluster Trace Exploration
**Goal:** Interactive EDA + model comparison for the Predictive Auto-Scaling project.

This notebook mirrors the CLI scripts but adds rich inline visualisations.
Requires the Codespace environment to be set up (`pip install -r requirements.txt`).

In [ ]:
import sys, subprocess
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
sns.set_theme(style='whitegrid', palette='muted')
print('✅  Imports OK')

## 1. Generate synthetic data (if not already present)

In [ ]:
data_path = ROOT / 'data' / 'instance_usage_sample.csv'
if not data_path.exists():
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'generate_synthetic.py')], check=True)
    print('Synthetic data generated.')
else:
    print(f'Data already exists: {data_path}')

## 2. Load and inspect the data

In [ ]:
from ingestion.data_loader import load_instance_usage
from config import DATETIME_COL, TARGET_COL

df = load_instance_usage(source='csv')
print(f'Shape: {df.shape}')
print(f'Date range: {df[DATETIME_COL].min()}  →  {df[DATETIME_COL].max()}')
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
df.set_index(DATETIME_COL)[TARGET_COL].plot(ax=ax, linewidth=0.5, alpha=0.8, color='#2196F3')
ax.set_title('Cluster CPU Usage – Full Series (NCU)', fontweight='bold')
ax.set_ylabel('CPU (NCU)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.tight_layout()
plt.show()

## 3. Statistical diagnostics (ADF, KPSS, STL)

In [ ]:
from analytics.diagnostics import run_full_diagnostics
diag = run_full_diagnostics(df)

from IPython.display import Image
Image(diag['plot_path'], width=900)

## 4. Feature engineering

In [ ]:
from analytics.feature_engineer import build_feature_matrix, time_split
from config import HORIZON

feat_df, feature_cols, target_cols = build_feature_matrix(df, horizon=HORIZON)
train_df, val_df, test_df = time_split(feat_df)

print(f'Features: {len(feature_cols)}')
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

## 5. Baseline models

In [ ]:
from models.baselines import compare_baselines

train_arr = train_df[TARGET_COL].values
test_arr  = test_df[TARGET_COL].values

baseline_results = compare_baselines(train_arr, test_arr, horizon=HORIZON, n_windows=30)
baseline_results

## 6. XGBoost forecaster

In [ ]:
from models.xgboost_forecaster import XGBoostForecaster

xgb = XGBoostForecaster(horizon=HORIZON)
xgb.fit(train_df, val_df, feature_cols, target_cols)

xgb_metrics = xgb.evaluate(test_df)
print('\nXGBoost metrics:')
xgb_metrics

In [ ]:
# Feature importance
fi = xgb.feature_importance(top_n=20)
fig, ax = plt.subplots(figsize=(8, 6))
fi.sort_values('importance').plot.barh(x='feature', y='importance', ax=ax, color='#FF9800')
ax.set_title('XGBoost Feature Importance (Top 20)', fontweight='bold')
ax.set_xlabel('Mean Gain')
plt.tight_layout()
plt.show()

## 7. Forecast visualisation

In [ ]:
from config import TARGET_COL as TC

# Plot one sample prediction vs actual
sample_idx = 100
row = test_df.iloc[[sample_idx]]
pred = xgb.predict(row[feature_cols])[0]
truth = row[[f'y_h{h}' for h in range(1, HORIZON+1)]].values[0]

fig, ax = plt.subplots(figsize=(10, 4))
x = range(1, HORIZON + 1)
ax.plot(x, truth, 'o-', label='Actual', color='#2196F3', linewidth=2)
ax.plot(x, pred,  's--', label='XGBoost', color='#F44336', linewidth=2)
ax.set_xlabel('Horizon (× 5 min)')
ax.set_ylabel('CPU (NCU)')
ax.set_title(f'Forecast vs Actual – H={HORIZON} ({HORIZON*5} min ahead)', fontweight='bold')
ax.legend()
ax.axhline(0.75, color='grey', linestyle=':', alpha=0.6, label='Scale-out threshold')
plt.tight_layout()
plt.show()

## 8. (Optional) Deep Learning – Informer / Autoformer
> ⚠️  Takes ~5 min on a standard Codespace CPU with default 300 steps.

In [ ]:
# Uncomment to run
# from models.transformer_model import TransformerForecaster, to_nixtla_format
# from analytics.feature_engineer import time_split
# 
# train_raw, val_raw, test_raw = time_split(df)
# train_nf = to_nixtla_format(train_raw)
# val_nf   = to_nixtla_format(val_raw)
# 
# fc = TransformerForecaster(model_name='informer')
# fc.fit(train_nf, val_nf)
# fc.save()
# print('Informer trained & saved.')

## 9. API demo

In [ ]:
import requests, json

# Ensure the API is running: uvicorn src.api.main:app --port 8000
BASE = 'http://localhost:8000'

try:
    resp = requests.get(f'{BASE}/health', timeout=2)
    print('API is running:', resp.json())

    payload = {
        'values': df[TARGET_COL].tail(50).tolist(),
        'model': 'xgboost'
    }
    resp = requests.post(f'{BASE}/scaling-decision', json=payload)
    decision = resp.json()
    print(json.dumps(decision, indent=2))
except requests.exceptions.ConnectionError:
    print('API not running. Start it with: uvicorn src.api.main:app --port 8000 --reload')